# Оптимальный транспорт (расстояние Вассерштейна)

**Проект «ИИ для учёных».** Практический блокнот к странице алгоритма «Оптимальный транспорт (расстояние Вассерштейна)».

## Научная цель

Оптимальный транспорт — математически обоснованный способ сравнивать два распределения с учётом геометрии пространства. Расстояние Вассерштейна измеряет минимальную «работу», необходимую для переноса одного распределения в другое: перемещение большого количества массы на большое расстояние стоит дорого, маленькой корректировки вблизи — дёшево.

В отличие от разности средних или χ²-критерия, расстояние Вассерштейна учитывает, куда именно «смещены» значения, а не только как изменились суммарные характеристики. Это делает его ценным инструментом для:

- сравнения профилей экспрессии генов или клеточных популяций из разных образцов;
- выравнивания облаков точек молекулярных конформаций без попарной разметки;
- доменной адаптации — «переноса» признакового пространства из одной экспериментальной установки в другую;
- построения геометрически осмысленных интерполяций между распределениями (барицентр Вассерштейна).

В этом блокноте мы вычислим точный транспортный план и энтропийно-регуляризованный план (алгоритм Синхорна), визуализируем перенос масс, сравним расстояние Вассерштейна с «наивными» мерами и покажем компромисс при разных параметрах регуляризации. Расчётное время прохождения — около шести-восьми минут, GPU не требуется.

## Интуиция за методом

Представьте две кучи песка разной формы. Расстояние Вассерштейна — это минимальный объём работы, который нужно совершить лопатой, чтобы переформировать одну кучу в другую: перемещение большого количества песка на большое расстояние стоит дорого, маленькой корректировки близлежащей части — дёшево. Обычная статистика смотрит лишь на то, «совпадают ли контуры куч», не замечая, что одна почти такая же, как другая, только чуть сдвинута.

Формально это задача **Монжа–Канторовича**: найти матрицу T[i,j] (транспортный план), где T[i,j] — сколько «массы» переместить из точки i источника в точку j цели. Суммарная стоимость ∑ T[i,j] · C[i,j] должна быть минимальной, где C[i,j] — стоимость переноса единицы массы (например, квадрат евклидова расстояния).

Точное решение задачи линейного программирования требует O(n³ log n) операций и неприменимо при n > 5 000. **Энтропийная регуляризация** (Кутюри, 2013) добавляет штраф за малую энтропию плана, превращая задачу в сильно выпуклую и решаемую итерационным **алгоритмом Синхорна**: поочерёдное масштабирование строк и столбцов матрицы. Параметр регуляризации ε управляет компромиссом: малый ε — план близок к оптимальному, но медленнее сходится; большой ε — план «размазывается», зато быстро и численно устойчиво.

**Ключевые понятия, которые встретятся в блокноте:**

- **Источник и цель** — два дискретных вероятностных распределения с весами (гистограммы).
- **Матрица стоимости** C[i,j] — попарные расстояния между точками источника и цели.
- **Транспортный план** T — матрица того, сколько массы откуда куда переносится.
- **Расстояние Вассерштейна** — минимальная суммарная стоимость транспортного плана; геометрически осмысленная мера сходства двух распределений.
- **Алгоритм Синхорна** — итерационный решатель для энтропийно-регуляризованной задачи.
- **Параметр регуляризации ε** — контролирует размытость плана и скорость сходимости.
- **Барицентр Вассерштейна** — «среднее» нескольких распределений в пространстве Вассерштейна; геометрически осмысленная интерполяция.

## 1. Установка библиотек

In [ ]:
%pip install -q pot==0.9.4 numpy==1.26.4 scipy==1.13.1 matplotlib==3.9.2 scikit-learn==1.5.1

## 2. Единый стиль графиков

Графики оформляются в едином визуальном стиле сайта «ИИ для учёных». Ниже встроено содержимое стилевого шаблона `notebooks/viz_style.py`.

In [ ]:
# Единый стилевой шаблон графиков проекта «ИИ для учёных».
# Значения синхронизированы с токенами темы сайта (styles/tokens.css).
VIZ_TOKENS = {
    "background": "#ffffff",
    "node_fill":  "#eef1f6",
    "node_text":  "#0e1726",
    "edge":       "#4d5e78",
    "grid":       "#dce2ec",
    "series":     ["#2563eb", "#0d9488", "#b45309", "#4d5e78"],
}
VIZ = VIZ_TOKENS


def apply_viz_style():
    """Настраивает matplotlib под единый стиль сайта."""
    import matplotlib as mpl
    from cycler import cycler
    t = VIZ_TOKENS
    mpl.rcParams.update({
        "font.family": "sans-serif",
        "font.sans-serif": ["Segoe UI", "DejaVu Sans", "Arial", "Helvetica"],
        "font.size": 12, "axes.titlesize": 15, "axes.titleweight": "semibold",
        "axes.labelsize": 13, "xtick.labelsize": 11, "ytick.labelsize": 11,
        "legend.fontsize": 11,
        "figure.facecolor": t["background"], "axes.facecolor": t["background"],
        "savefig.facecolor": t["background"], "figure.dpi": 110, "savefig.dpi": 150,
        "axes.prop_cycle": cycler(color=t["series"]),
        "text.color": t["node_text"], "axes.labelcolor": t["node_text"],
        "axes.titlecolor": t["node_text"], "axes.edgecolor": t["edge"],
        "xtick.color": t["edge"], "ytick.color": t["edge"], "axes.linewidth": 1.2,
        "axes.grid": True, "grid.color": t["grid"], "grid.linewidth": 1.0,
        "grid.linestyle": "-", "axes.axisbelow": True,
        "lines.linewidth": 2.0, "lines.markersize": 6, "patch.linewidth": 1.5,
        "axes.spines.top": False, "axes.spines.right": False,
        "legend.frameon": False,
    })


def get_palette(n=None):
    """Возвращает список цветов рядов из токенов темы."""
    series = VIZ_TOKENS["series"]
    if n is None:
        return list(series)
    return [series[i % len(series)] for i in range(n)]


apply_viz_style()
print("Единый стиль графиков подключён.")

## 3. Демонстрационные данные

Создадим два синтетических двумерных распределения точек — «источник» и «цель». Источник — смесь двух гауссовых облаков в форме буквы «L», цель — смесь двух облаков, сдвинутых и повёрнутых. Такая геометрия наглядно показывает, что оптимальный транспорт «понимает» пространственное расположение масс, а не просто сравнивает гистограммы.

Для равномерного сравнения каждому распределению назначены равные веса (гистограмма без предпочтений). Зафиксируем seed для воспроизводимости.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

RNG = np.random.default_rng(42)

# Источник: два облака (верхнее левое и нижнее левое).
n = 40  # точек на каждое облако
X_src = np.vstack([
    RNG.normal(loc=[-2.0,  2.0], scale=0.5, size=(n, 2)),
    RNG.normal(loc=[-2.0, -1.0], scale=0.5, size=(n, 2)),
])

# Цель: два облака (правое верхнее и правое нижнее, иные центры).
X_tgt = np.vstack([
    RNG.normal(loc=[ 2.0,  1.5], scale=0.5, size=(n, 2)),
    RNG.normal(loc=[ 1.0, -2.0], scale=0.5, size=(n, 2)),
])

n_src = len(X_src)
n_tgt = len(X_tgt)

# Равномерные веса: каждая точка несёт одинаковую долю «массы».
a = np.ones(n_src) / n_src   # вес источника
b = np.ones(n_tgt) / n_tgt   # вес цели

print(f"Источник: {n_src} точек, цель: {n_tgt} точек")
print(f"Суммарный вес источника: {a.sum():.4f}, цели: {b.sum():.4f}")

# Визуализация исходных данных.
fig, ax = plt.subplots(figsize=(7, 5.5))
ax.scatter(X_src[:, 0], X_src[:, 1], s=40, color=VIZ["series"][0],
           alpha=0.85, label="Источник (μ)")
ax.scatter(X_tgt[:, 0], X_tgt[:, 1], s=40, color=VIZ["series"][1],
           alpha=0.85, marker="^", label="Цель (ν)")
ax.set_title("Исходные распределения")
ax.set_xlabel("Признак 1")
ax.set_ylabel("Признак 2")
ax.legend(loc="upper right")
fig.tight_layout()
plt.show()

## 4. Применение метода

### Шаг 4.1. Матрица стоимости

Матрица стоимости C[i,j] определяет, сколько «стоит» перемещение единицы массы из точки i источника в точку j цели. Стандартный выбор для пространственных данных — квадрат евклидова расстояния, тогда расстояние Вассерштейна называют расстоянием Вассерштейна-2.

Масштаб матрицы C влияет на интерпретируемость параметра регуляризации ε: принято нормировать C на медианное квадратичное расстояние, чтобы ε можно было задавать в относительных единицах.

In [ ]:
import ot

# Матрица квадратичной евклидовой стоимости (n_src × n_tgt).
C = ot.dist(X_src, X_tgt, metric="sqeuclidean")

# Нормировка: делим на медиану, чтобы ε имел относительный масштаб.
C_norm = C / np.median(C)

print(f"Размер матрицы стоимости: {C.shape}")
print(f"Минимальная стоимость: {C.min():.4f}")
print(f"Медианная стоимость:   {np.median(C):.4f}")
print(f"Максимальная стоимость: {C.max():.4f}")

# Тепловая карта матрицы стоимости.
fig, ax = plt.subplots(figsize=(6.5, 5.2))
im = ax.imshow(C, aspect="auto", cmap="Blues",
               origin="upper")
cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label("Квадрат евклидова расстояния")
ax.set_title("Матрица стоимости C[i, j]")
ax.set_xlabel("Точки цели j")
ax.set_ylabel("Точки источника i")
fig.tight_layout()
plt.show()

**Как читать тепловую карту стоимости:**

Строки соответствуют точкам источника, столбцы — точкам цели. Тёмная клетка (i, j) означает, что перевозить массу из точки i в точку j дорого (они далеко друг от друга). Светлая клетка — дёшево. Оптимальный план будет «избегать» тёмных клеток и предпочитать светлые — переносить массу по коротким маршрутам.

Блочная структура тепловой карты отражает исходную геометрию: первые 40 строк (первое облако источника) имеют свои «дешёвые» столбцы в одном блоке цели, вторые 40 строк — в другом.

### Шаг 4.2. Точный транспортный план (алгоритм EMD)

`ot.emd` решает задачу линейного программирования точно — это **Earth Mover's Distance** (расстояние Вассерштейна-1 при единичной стоимости, или Вассерштейна-2 при квадратичной). Для n = 80 это быстро; при n > 3 000–5 000 точный решатель становится непрактичным.

In [ ]:
# Точный транспортный план (решатель EMD / линейное программирование).
T_emd = ot.emd(a, b, C_norm)

# Расстояние Вассерштейна: суммарная стоимость оптимального плана
# (в единицах нормированной матрицы стоимости).
W2_emd = np.sum(T_emd * C_norm)
print(f"Точное расстояние Вассерштейна (W^2, нормированное): {W2_emd:.6f}")

# Для сравнения — «наивная» мера: разность центроидов.
diff_means = np.linalg.norm(X_src.mean(axis=0) - X_tgt.mean(axis=0))
print(f"Наивная мера — расстояние между средними:            {diff_means:.4f}")

print(f"\nТранспортный план T_emd: форма {T_emd.shape}, "
      f"ненулевых элементов {np.sum(T_emd > 1e-10)}")

### Шаг 4.3. Визуализация точного транспортного плана

Транспортный план — матрица T[i,j]. Чтобы увидеть его геометрически, нарисуем линии между каждой точкой источника и каждой точкой цели, которые связаны ненулевым потоком. Толщина линии пропорциональна потоку T[i,j].

In [ ]:
def plot_transport_plan(ax, X_s, X_t, T_plan, title,
                        threshold=5e-5, max_lw=4.0):
    """
    Рисует транспортный план линиями между точками источника и цели.
    Толщина линии пропорциональна потоку T[i,j].
    """
    T_max = T_plan.max()
    for i in range(len(X_s)):
        for j in range(len(X_t)):
            flow = T_plan[i, j]
            if flow < threshold:
                continue
            lw = max_lw * flow / T_max
            ax.plot([X_s[i, 0], X_t[j, 0]],
                    [X_s[i, 1], X_t[j, 1]],
                    color=VIZ["edge"], linewidth=lw, alpha=0.55, zorder=1)
    ax.scatter(X_s[:, 0], X_s[:, 1], s=45, color=VIZ["series"][0],
               zorder=3, label="Источник (μ)")
    ax.scatter(X_t[:, 0], X_t[:, 1], s=45, color=VIZ["series"][1],
               marker="^", zorder=3, label="Цель (ν)")
    ax.set_title(title)
    ax.set_xlabel("Признак 1")
    ax.set_ylabel("Признак 2")
    ax.legend(loc="upper right")


fig, ax = plt.subplots(figsize=(8, 6))
plot_transport_plan(ax, X_src, X_tgt, T_emd,
                    "Точный транспортный план (EMD)")
fig.tight_layout()
plt.show()

**Как читать этот график:**

Каждая линия — маршрут переноса «массы» из точки источника (синий круг) в точку цели (зелёный треугольник). Толщина линии пропорциональна количеству переносимой массы: жирная линия — большой поток, тонкая — малый.

Точный EMD-план **разреженный**: каждая точка источника связана лишь с несколькими точками цели. Это математическое свойство решения задачи линейного программирования — угловое решение всегда разреженное (не более n_src + n_tgt - 1 ненулевых элементов).

Обратите внимание, как план «уважает» геометрию: масса из верхнего левого облака перетекает преимущественно в правое верхнее, а не хаотично в любую точку цели.

### Шаг 4.4. Энтропийно-регуляризованный план (алгоритм Синхорна)

`ot.sinkhorn` решает регуляризованную задачу итерационным масштабированием строк и столбцов. Параметр `reg` — это ε. Попробуем три значения: малое (план близок к точному), среднее и большое (план сильно размазан).

In [ ]:
import warnings

# Три значения параметра регуляризации ε.
eps_values = [0.01, 0.1, 1.0]
plans_sinkhorn = {}
costs_sinkhorn = {}

for eps in eps_values:
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        T_sh = ot.sinkhorn(a, b, C_norm, reg=eps,
                           numItermax=2000, stopThr=1e-9)
    plans_sinkhorn[eps] = T_sh
    costs_sinkhorn[eps] = np.sum(T_sh * C_norm)
    print(f"ε = {eps:.2f}:  стоимость плана = {costs_sinkhorn[eps]:.6f}  "
          f"(ненулевых элементов > 1e-6: {np.sum(T_sh > 1e-6)})")

print(f"\nТочный EMD:    стоимость плана = {W2_emd:.6f}  "
      f"(ненулевых элементов > 1e-10: {np.sum(T_emd > 1e-10)})")

### Шаг 4.5. Сравнение планов при разных ε

На трёх панелях покажем, как регуляризация меняет структуру транспортного плана: от «острого» (близкого к точному EMD) до полностью размытого.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5.2), sharey=True)

thresholds = {0.01: 5e-5, 0.1: 1e-4, 1.0: 2e-4}

for ax, eps in zip(axes, eps_values):
    T_sh = plans_sinkhorn[eps]
    cost = costs_sinkhorn[eps]
    plot_transport_plan(ax, X_src, X_tgt, T_sh,
                        f"Синхорн, ε = {eps}\nстоимость = {cost:.4f}",
                        threshold=thresholds[eps], max_lw=3.5)

axes[1].legend().remove()   # убираем дублирующиеся легенды
axes[2].legend().remove()

fig.suptitle(
    "Транспортный план алгоритма Синхорна при разных параметрах регуляризации ε",
    fontsize=13, y=1.02
)
fig.tight_layout()
plt.show()

**Как читать этот график:**

- **Малое ε (0.01):** план почти совпадает с точным EMD — линии толстые, направленные, каждая точка источника связана с 1–2 точками цели. Планов много потому, что малый ε делает задачу численно трудной (медленная сходимость).
- **Среднее ε (0.1):** план заметно размыт — каждая точка источника посылает потоки в несколько точек цели, причём и в «неоптимальные» по расстоянию. Стоимость плана немного выше точного минимума.
- **Большое ε (1.0):** план приближается к независимому произведению мер a ⊗ b — каждая точка источника почти равномерно соединена со всеми точками цели. Транспортный план теряет содержательную геометрическую интерпретацию, но вычисляется очень быстро.

**Вывод:** выбирайте ε в диапазоне 0.01–0.1 от медианного квадратичного расстояния в вашей задаче; проверяйте сходимость (библиотека POT сигнализирует предупреждением при превышении числа итераций).

### Шаг 4.6. Компромисс ε: стоимость плана и «размытость»

Покажем на одном графике: как стоимость плана и энтропия транспортного плана меняются при варьировании ε.

In [ ]:
eps_grid = np.logspace(-2, 0.5, 18)  # от 0.01 до ~3.2
costs_grid = []
entropies_grid = []

for eps in eps_grid:
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        T_sh = ot.sinkhorn(a, b, C_norm, reg=eps,
                           numItermax=3000, stopThr=1e-9)
    costs_grid.append(np.sum(T_sh * C_norm))
    # Энтропия транспортного плана: -sum(T * log(T + tiny)).
    T_pos = T_sh + 1e-300
    entropies_grid.append(-np.sum(T_pos * np.log(T_pos)))

fig, axes = plt.subplots(1, 2, figsize=(13.5, 5.0))

# Левый: стоимость плана vs ε.
axes[0].semilogx(eps_grid, costs_grid, marker="o", markersize=5,
                 color=VIZ["series"][0], label="Синхорн")
axes[0].axhline(W2_emd, color=VIZ["series"][2], linestyle="--",
                label=f"Точный EMD: {W2_emd:.4f}")
axes[0].set_title("Стоимость плана vs параметр регуляризации ε")
axes[0].set_xlabel("ε (логарифмическая шкала)")
axes[0].set_ylabel("Суммарная стоимость транспортного плана")
axes[0].legend(loc="upper left")

# Правый: энтропия плана vs ε.
axes[1].semilogx(eps_grid, entropies_grid, marker="o", markersize=5,
                 color=VIZ["series"][1])
axes[1].set_title("«Размытость» плана (энтропия) vs параметр регуляризации ε")
axes[1].set_xlabel("ε (логарифмическая шкала)")
axes[1].set_ylabel("Энтропия транспортного плана H(T)")

fig.tight_layout()
plt.show()

**Как читать эти графики:**

- **Левый (стоимость vs ε):** при малом ε стоимость плана Синхорна стремится к значению точного EMD (пунктирная линия). При росте ε стоимость плана возрастает — регуляризация вынуждает план посылать массу по «неоптимальным» маршрутам.
- **Правый (энтропия vs ε):** энтропия H(T) монотонно растёт с ε — план становится всё более равномерным (размытым). При ε → 0 план вырождается в разреженный (низкая энтропия); при ε → ∞ — в равномерное произведение мер (максимальная энтропия).

**Практический вывод:** для задач, где важна точность расстояния, выбирайте малое ε (≈0.01–0.05); для задач, где требуется лишь мягкий план для последующей оптимизации или функции потерь нейросети, допустимы бо́льшие ε.

### Шаг 4.7. Сравнение с «наивными» мерами сходства

Покажем принципиальное различие между расстоянием Вассерштейна и «наивными» мерами (разность средних, разность медиан) на искусственном примере: два распределения с одинаковыми средними, но разной формой.

In [ ]:
from scipy.stats import wasserstein_distance

rng2 = np.random.default_rng(7)

# Три одномерных распределения с одинаковым средним (≈0).
dist_A = rng2.normal(0.0, 1.0, 500)           # узкое нормальное
dist_B = rng2.normal(0.0, 3.0, 500)           # широкое нормальное (то же среднее!)
dist_C = np.concatenate([                     # двумодальное (то же среднее!)
    rng2.normal(-3.0, 0.5, 250),
    rng2.normal( 3.0, 0.5, 250),
])

distributions = {"A: N(0, 1)": dist_A,
                 "B: N(0, 3)": dist_B,
                 "C: двумодальное": dist_C}

print("Средние:")
for name, d in distributions.items():
    print(f"  {name}: среднее = {d.mean():.3f}, медиана = {np.median(d):.3f}")

print("\nРасстояния Вассерштейна-1 от A:")
for name, d in list(distributions.items())[1:]:
    w1 = wasserstein_distance(dist_A, d)
    diff_mean = abs(dist_A.mean() - d.mean())
    diff_median = abs(np.median(dist_A) - np.median(d))
    print(f"  A vs {name}:")
    print(f"    W1 = {w1:.4f}  |  |Δсредних| = {diff_mean:.4f}  |  |Δмедиан| = {diff_median:.4f}")

# Визуализация трёх распределений.
bins = np.linspace(-8, 8, 60)
colors = get_palette(3)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.8), sharey=True)
for ax, (name, d), color in zip(axes, distributions.items(), colors):
    ax.hist(d, bins=bins, color=color, alpha=0.75, density=True,
            edgecolor=VIZ["background"], linewidth=0.4)
    ax.axvline(d.mean(), color=VIZ["node_text"], linestyle="--",
               linewidth=1.5, label=f"Среднее: {d.mean():.2f}")
    ax.set_title(name)
    ax.set_xlabel("Значение")
    ax.legend(loc="upper right")

axes[0].set_ylabel("Плотность")
fig.suptitle(
    "Три распределения с одинаковым средним — разное расстояние Вассерштейна",
    fontsize=13
)
fig.tight_layout()
plt.show()

**Как читать эти результаты:**

Все три распределения имеют практически одинаковое среднее (≈0). Разность средних не отличила бы их. Расстояние Вассерштейна-1 чётко разделяет их:

- A vs B: W1 ≫ 0, хотя |Δсредних| ≈ 0. Широкое нормальное «далеко» от узкого в смысле необходимой работы по переносу массы.
- A vs C: W1 ещё больше — двумодальное распределение требует перевезти массу к двум полюсам.

Это ключевое преимущество оптимального транспорта: он обнаруживает различия в форме распределений, которые теряются при сравнении только через моменты.

## 5. Барицентр Вассерштейна: интерполяция между распределениями

Барицентр Вассерштейна — «среднее» нескольких распределений в пространстве оптимального транспорта. В отличие от пиксельного или гистограммного среднего, он геометрически переносит форму, а не усредняет её. Покажем одномерную интерполяцию между двумя гауссианами.

In [ ]:
# Одномерная интерполяция через барицентр Вассерштейна.
# Используем дискретное представление: гистограммы на общей сетке.

x_grid = np.linspace(-5, 8, 200)
dx = x_grid[1] - x_grid[0]

# Два «крайних» распределения: левая узкая и правая широкая гауссианы.
from scipy.stats import norm

p_left  = norm.pdf(x_grid, loc=-2.0, scale=0.8); p_left  /= p_left.sum()
p_right = norm.pdf(x_grid, loc= 5.0, scale=1.5); p_right /= p_right.sum()

# Интерполяция: барицентр при весах (1-t) · p_left + t · p_right
# в пространстве Вассерштейна реализуется через функцию ot.lp.barycenter.
alphas_interp = [0.0, 0.25, 0.5, 0.75, 1.0]

M_bary = ot.utils.dist(x_grid.reshape(-1, 1), x_grid.reshape(-1, 1),
                        metric="sqeuclidean")
M_bary /= M_bary.max()

fig, ax = plt.subplots(figsize=(9, 5.5))
colors_interp = [VIZ["series"][0], "#4a90d9", VIZ["series"][3],
                 "#6dbfa5", VIZ["series"][1]]

for alpha, color in zip(alphas_interp, colors_interp):
    weights = np.array([1 - alpha, alpha])
    histograms = np.column_stack([p_left, p_right])
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        bary = ot.lp.barycenter(histograms, M_bary,
                                weights=weights, solver="interior-point")
    bary = np.abs(bary); bary /= bary.sum()
    label = (f"t = {alpha:.2f}"
             + (" (источник)" if alpha == 0.0
                else " (цель)" if alpha == 1.0 else ""))
    lw = 2.5 if alpha in (0.0, 1.0) else 1.8
    ax.plot(x_grid, bary / dx, color=color, linewidth=lw,
            label=label, alpha=0.9)

ax.set_title("Интерполяция между двумя распределениями\nчерез барицентр Вассерштейна")
ax.set_xlabel("Значение")
ax.set_ylabel("Плотность")
ax.legend(loc="upper right")
fig.tight_layout()
plt.show()

**Как читать этот график:**

Кривые показывают интерполяцию от источника (t = 0.00, синий, узкий пик слева) до цели (t = 1.00, зелёный, широкий пик справа). Промежуточные барицентры (t = 0.25, 0.50, 0.75) плавно сдвигают и расширяют пик — геометрически осмысленная интерполяция.

Обратите внимание: барицентр Вассерштейна не строит «среднюю гистограмму» (что дало бы двугорбое распределение с провалом посередине), а именно **переносит** форму по прямому пути в пространстве распределений. Это и есть ключевое преимущество метода для усреднения клеточных популяций, форм молекул, пространственных карт.

## 6. Интерпретация результата

**Что означают числа и графики:**

- **Расстояние Вассерштейна (W²)** — минимальная суммарная стоимость переноса всей массы из источника в цель. Меньшее значение — распределения «ближе» геометрически. Оно не зависит от выбора разбиения на бины (как χ²), но зависит от выбора функции стоимости C[i,j].
- **Тепловая карта плана T[i,j]** — яркая клетка (i,j) означает, что значительная доля массы из точки i перетекает в точку j. Разреженный план (EMD) означает «экономичную» транспортировку; плотный (большое ε) — «мягкую».
- **Транспортные линии** — читайте как «маршруты перевозки»; жирная линия = большой поток. Геометрическая интерпретация: масса «идёт» по кратчайшим доступным путям.
- **Компромисс ε:** стоимость плана Синхорна всегда ≥ стоимости EMD; при ε → 0 они совпадают. Если кривая стоимости при малом ε не сошлась к значению EMD — увеличьте `numItermax` или используйте `method='sinkhorn_log'` для численной устойчивости.
- **Барицентр Вассерштейна** — при t = 0.5 получаем геометрическое «среднее»; в отличие от арифметического среднего, оно не создаёт артефактных пиков там, где данных нет.

**Сравнение с наивными мерами:** разность средних нечувствительна к форме распределения — два совершенно разных распределения могут иметь одинаковые средние. Расстояние Вассерштейна учитывает всю геометрию, а не только моменты.

## 7. Попробуйте на своих данных

Подготовьте два набора точек (или гистограммы): источник и цель. Убедитесь, что размер не превышает 3 000–5 000 точек на каждое распределение для точного решателя; для большего размера используйте Синхорн с разумным ε.

1. Загрузите файл в Colab: панель слева, вкладка «Файлы», кнопка загрузки.
2. Снимите комментарии в ячейке ниже и укажите свои данные.
3. Скорректируйте функцию стоимости (`metric`) под природу ваших объектов: для пространственных данных — `sqeuclidean`; для нормированных дескрипторов — `cosine` или `cityblock`.
4. Последовательно выполните ячейки разделов 4 и 5.

In [ ]:
# --- Шаблон загрузки своих данных ---
# import pandas as pd
#
# df_src = pd.read_csv("my_source.csv")   # строки — точки, столбцы — признаки
# df_tgt = pd.read_csv("my_target.csv")
#
# X_src = df_src.to_numpy(dtype=float)
# X_tgt = df_tgt.to_numpy(dtype=float)
#
# # Равномерные веса (можно задать свои: a = df_src["weight"].to_numpy()).
# a = np.ones(len(X_src)) / len(X_src)
# b = np.ones(len(X_tgt)) / len(X_tgt)
#
# # Матрица стоимости и нормировка.
# C = ot.dist(X_src, X_tgt, metric="sqeuclidean")
# C_norm = C / np.median(C)
#
# # Точный план (только если n < 3000–5000).
# T_emd = ot.emd(a, b, C_norm)
# W2 = np.sum(T_emd * C_norm)
# print(f"Расстояние Вассерштейна (нормированное): {W2:.6f}")
#
# # Синхорн с регуляризацией (для больших n).
# # T_sh = ot.sinkhorn(a, b, C_norm, reg=0.05, numItermax=2000)
# print(f"Размеры источника: {X_src.shape}, цели: {X_tgt.shape}")

## Поэкспериментируйте сами

На основном примере (раздел 4) попробуйте:

| Что изменить | Где | Что наблюдать |
|---|---|---|
| `n = 100` вместо 40 | Раздел 3 | Становится ли план «чище»? Вырастает ли время EMD? |
| `metric="cityblock"` вместо `"sqeuclidean"` | Шаг 4.1 | Как меняется форма плана при другой метрике? |
| `reg=0.005` в Синхорне | Шаг 4.4 | Появляются ли предупреждения о несходимости? |
| Смещение облаков источника ближе к цели | Раздел 3 | Как уменьшается W²? Меняется ли структура плана? |
| Добавить третье облако источника | Раздел 3 | Как распределяются потоки при трёх группах vs двух? |

Дополнительный эксперимент: замените в разделе 4.7 `dist_C` на распределение, совпадающее с `dist_A` по форме, но сдвинутое вправо на 2. Сравните W1 и |Δсредних|. Убедитесь, что расстояние Вассерштейна отразило сдвиг правильно, тогда как разность медиан тоже сработала, но не отразила бы изменение формы.

## 8. Краткие выводы

- Расстояние Вассерштейна измеряет минимальную суммарную «работу» по переносу одного распределения в другое; оно учитывает геометрию пространства, а не только моменты.
- Точный решатель EMD (`ot.emd`) даёт разреженный план — оптимальный по стоимости, но применимый лишь при n ≤ 3 000–5 000 точек.
- Алгоритм Синхорна (`ot.sinkhorn`) с энтропийной регуляризацией ε масштабируется на большие n и реализуется как серия умножений матрица-вектор; цена — небольшое отклонение плана от оптимального.
- Параметр ε управляет компромиссом: малый ε → точнее план, медленнее и численно жёстче; большой ε → быстрее, но план «размыт».
- Нормировка матрицы стоимости на медиану является хорошей практикой: это делает ε интерпретируемым в относительных единицах.
- Барицентр Вассерштейна даёт геометрически осмысленную интерполяцию между распределениями — в отличие от арифметического среднего гистограмм.
- Библиотека POT предоставляет готовые реализации всех перечисленных методов; для дифференцируемых вариантов (обучение нейросетей) используйте GeomLoss (PyTorch) или OTT-JAX.

## Три вопроса для самопроверки

Ответьте до того, как раскроете подсказку, — это проверяет, что метод понят, а не просто запущен.

1. Вы вычислили расстояние Вассерштейна между двумя клеточными популяциями (по 500 клеток каждая, 50 признаков экспрессии). Точный решатель EMD работает 8 секунд, что приемлемо. Однако при добавлении третьего образца (1 000 клеток) время вырастает до нескольких минут. Объясните причину и предложите практическое решение.

2. Вы применили алгоритм Синхорна с ε = 0.001 и получили предупреждение о несходимости. При ε = 0.5 предупреждений нет, но стоимость плана заметно выше стоимости EMD. Какую стратегию выбора ε вы применили бы и как проверили бы качество полученного плана?

3. Два исследователя сравнивают одни и те же два облака точек молекулярных конформаций. Первый использует евклидову матрицу стоимости, второй — матрицу косинусного расстояния. Они получают разные расстояния Вассерштейна. Кто из них «прав» и что именно определяет правильный выбор функции стоимости?

<details>
<summary>Показать ориентиры для ответов</summary>

1. Точный решатель EMD имеет вычислительную сложность O(n³ log n): удвоение числа точек увеличивает время примерно в 8–16 раз. При n = 1 000 это несколько минут на CPU. Практические решения: перейти на алгоритм Синхорна (`ot.sinkhorn` с разумным ε); использовать субвыборку — случайно отбирать 200–500 клеток из каждого образца при каждом вычислении; применить проекционную аппроксимацию (Sliced Wasserstein Distance) с O(n log n) стоимостью; или снизить размерность признакового пространства перед вычислением.

2. Стратегия настройки ε: начните с медианного значения квадратичных расстояний в C и возьмите ε ≈ 0.01–0.05 от него (это соответствует «умеренной» регуляризации). Увеличьте `numItermax` до 5 000–10 000 и попробуйте `method='sinkhorn_log'` — логарифмический домен устойчивее при малых ε. Проверка качества: (а) убедитесь, что предупреждений о несходимости нет; (б) сравните стоимость Синхорна со стоимостью EMD на подвыборке — разрыв не должен превышать 5–10%; (в) проверьте, что маргинальные суммы плана T (по строкам и столбцам) близки к заданным весам a и b.

3. Ни один не «прав» автоматически — выбор функции стоимости является содержательным решением исследователя. Евклидова метрика уместна, когда конформации представлены координатами атомов и физический смысл «близости» — пространственное расстояние между атомами. Косинусное расстояние уместно для нормированных дескрипторов (например, ECFP-отпечатков или эмбеддингов), где важна угловая близость, а не абсолютная величина вектора. Правильный выбор определяется тем, что в данной задаче означает «дорого перевозить»: если физический смысл — реальная работа по деформации молекулы, нужна геодезическая метрика в конформационном пространстве, а не евклидова.
</details>